# Naive Bayes

## Learning Objectives
1. Derive and implement Gaussian Naive Bayes from scratch using NumPy.
2. Compare GaussianNB, MultinomialNB, and BernoulliNB with scikit-learn.
3. Build a spam detection pipeline with MultinomialNB and TF-IDF features.
4. Benchmark Naive Bayes against Logistic Regression on text and tabular data.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.datasets import make_classification
from sklearn.preprocessing import MinMaxScaler

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1: Gaussian Naive Bayes from Scratch (NumPy)


In [ ]:
class GaussianNBScratch:
    """Gaussian Naive Bayes: P(y|x) proportional to P(y) * prod_j P(x_j|y)."""

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'GaussianNBScratch':
        self.classes_ = np.unique(y)
        self.priors_ = {}
        self.means_ = {}
        self.vars_ = {}
        for c in self.classes_:
            X_c = X[y == c]
            self.priors_[c] = np.log(len(X_c) / len(X))
            self.means_[c] = X_c.mean(axis=0)
            self.vars_[c] = X_c.var(axis=0) + 1e-9
        return self

    def _log_likelihood(self, x: np.ndarray, c) -> float:
        mu, var = self.means_[c], self.vars_[c]
        return -0.5 * np.sum(np.log(2 * np.pi * var) + (x - mu) ** 2 / var)

    def predict(self, X: np.ndarray) -> np.ndarray:
        preds = []
        for x in X:
            scores = {c: self.priors_[c] + self._log_likelihood(x, c)
                      for c in self.classes_}
            preds.append(max(scores, key=scores.get))
        return np.array(preds)

    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        return (self.predict(X) == y).mean()


X_gnb, y_gnb = make_classification(n_samples=500, n_features=10, n_classes=3,
                                    n_informative=6, n_redundant=2, random_state=42)
split = 400
gnb = GaussianNBScratch().fit(X_gnb[:split], y_gnb[:split])
acc_scratch = gnb.score(X_gnb[split:], y_gnb[split:])

gnb_sk = GaussianNB().fit(X_gnb[:split], y_gnb[:split])
acc_sklearn = gnb_sk.score(X_gnb[split:], y_gnb[split:])

print(f"Scratch GaussianNB accuracy : {acc_scratch:.4f}")
print(f"Sklearn GaussianNB accuracy : {acc_sklearn:.4f}")
print(f"Accuracy delta              : {abs(acc_scratch - acc_sklearn):.4f}")

print("\nClass means (first 3 features):")
for c in gnb.classes_:
    print(f"  Class {c}: {gnb.means_[c][:3]}")


## Level 2: MultinomialNB / GaussianNB / BernoulliNB Comparison (sklearn)


In [ ]:
try:
    from sklearn.datasets import fetch_20newsgroups
    categories = ['sci.space', 'rec.sport.hockey',
                  'talk.politics.misc', 'comp.graphics']
    news_train = fetch_20newsgroups(subset='train', categories=categories,
                                     remove=('headers', 'footers', 'quotes'))
    news_test  = fetch_20newsgroups(subset='test',  categories=categories,
                                     remove=('headers', 'footers', 'quotes'))
    X_text_tr, y_text_tr = news_train.data, news_train.target
    X_text_te, y_text_te = news_test.data,  news_test.target
    print(f"Loaded 20newsgroups: {len(X_text_tr)} train, {len(X_text_te)} test")
except Exception:
    print("Using synthetic fallback (offline)")
    X_text_tr = [f"word{i} sample text class{i%4}" for i in range(800)]
    y_text_tr = np.array([i % 4 for i in range(800)])
    X_text_te  = [f"word{i} test text class{i%4}" for i in range(200)]
    y_text_te  = np.array([i % 4 for i in range(200)])

from sklearn.preprocessing import FunctionTransformer

pipelines = {
    "MultinomialNB (TF-IDF)": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=5000, sublinear_tf=True)),
        ("clf", MultinomialNB()),
    ]),
    "BernoulliNB (binary)": Pipeline([
        ("cv", CountVectorizer(max_features=5000, binary=True)),
        ("clf", BernoulliNB()),
    ]),
    "GaussianNB (TF-IDF dense)": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=500)),
        ("todense", FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)),
        ("clf", GaussianNB()),
    ]),
}

print(f"\n{'Pipeline':>32} | {'Test Acc':>9}")
print("-" * 44)
for name, pipe in pipelines.items():
    try:
        pipe.fit(X_text_tr, y_text_tr)
        acc_pipe = pipe.score(X_text_te, y_text_te)
        print(f"{name:>32} | {acc_pipe:>9.4f}")
    except Exception as exc:
        if "out of memory" in str(exc).lower():
            print(f"{name:>32} | OOM: reduce max_features")
        else:
            print(f"{name:>32} | ERROR: {str(exc)[:30]}")


## Real-World Example 1: Spam Detection with MultinomialNB


In [ ]:
SPAM_TEMPLATES = [
    "Win a free iPhone click here now",
    "Congratulations you have been selected for prize money",
    "Buy cheap pills online discount offer",
    "Make money fast working from home guaranteed",
    "Urgent your account has been compromised verify now",
]
HAM_TEMPLATES = [
    "Meeting scheduled for tomorrow at 10am please confirm",
    "The project report is due on Friday please review",
    "Could you please send me the quarterly analysis",
    "Thanks for your email I will follow up this week",
    "Please review the attached document and provide feedback",
]

rng = np.random.default_rng(42)
emails, labels = [], []
for _ in range(1000):
    if rng.random() < 0.4:
        base = rng.choice(SPAM_TEMPLATES)
        noise = " ".join(rng.choice(["deal", "money", "free", "click", "win"])
                         for _ in range(rng.integers(2, 6)))
        emails.append(base + " " + noise); labels.append(1)
    else:
        base = rng.choice(HAM_TEMPLATES)
        noise = " ".join(rng.choice(["meeting", "report", "review", "team"])
                         for _ in range(rng.integers(1, 4)))
        emails.append(base + " " + noise); labels.append(0)

labels = np.array(labels)
spam_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=3000, sublinear_tf=True)),
    ("clf", MultinomialNB(alpha=0.5)),
])

cv_spam = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_spam = cross_val_score(spam_pipeline, emails, labels, cv=cv_spam, scoring='f1')
print(f"Spam detection F1 (5-fold CV): {scores_spam.mean():.4f} +/- {scores_spam.std():.4f}")

spam_pipeline.fit(emails, labels)
test_emails = [
    "Win a free prize click here",
    "Please review the attached report",
]
preds = spam_pipeline.predict(test_emails)
probs = spam_pipeline.predict_proba(test_emails)
for email, pred, prob in zip(test_emails, preds, probs):
    label = "SPAM" if pred == 1 else "HAM"
    print(f"[{label}] P(spam)={prob[1]:.3f}: '{email[:50]}'")


## Real-World Example 2: Laplace Smoothing Effect on Sparse Features


In [ ]:
def make_sparse_text_dataset(n=500, vocab_size=200, avg_words=5):
    """Generate sparse bag-of-words with partial vocabulary overlap."""
    rng2 = np.random.default_rng(0)
    texts, labels_s = [], []
    for i in range(n):
        c = i % 2
        vocab_start = 0 if c == 0 else vocab_size // 2
        vocab_end = vocab_size // 2 if c == 0 else vocab_size
        n_words = rng2.integers(2, avg_words + 3)
        word_ids = rng2.integers(vocab_start, vocab_end, size=n_words)
        texts.append(" ".join(f"w{w}" for w in word_ids))
        labels_s.append(c)
    return texts, np.array(labels_s)


texts_s, labels_s = make_sparse_text_dataset(n=600)
alphas = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]

print(f"{'Alpha':>8} | {'CV Accuracy':>12} | {'Notes':>20}")
print("-" * 50)
for alpha in alphas:
    pipe_s = Pipeline([
        ("cv", CountVectorizer(max_features=200)),
        ("clf", MultinomialNB(alpha=alpha)),
    ])
    try:
        cv_s = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        sc = cross_val_score(pipe_s, texts_s, labels_s, cv=cv_s, scoring='accuracy')
        note = "optimal" if 0.1 <= alpha <= 1.0 else "over-smooth"
        print(f"{alpha:>8.3f} | {sc.mean():>12.4f} | {note:>20}")
    except Exception as e:
        print(f"{alpha:>8.3f} | {'ERROR':>12} | {str(e)[:20]:>20}")


## Real-World Example 3: Naive Bayes vs Logistic Regression Comparison


In [ ]:
comparisons = {}

for size in [200, 2000]:
    key = f"Text n={size}"
    txt_s = [f"word{np.random.randint(0, 100)} sample{np.random.randint(0, 50)}"
             for _ in range(size)]
    lbl_s = np.array([i % 2 for i in range(size)])
    comparisons[key] = (txt_s, lbl_s, "text")

for size in [1000, 200]:
    key = f"Tabular n={size}"
    X_t, y_t = make_classification(n_samples=size, n_features=20,
                                    n_informative=8, random_state=42)
    comparisons[key] = (X_t, y_t, "tabular")

cv_comp = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"{'Dataset':>20} | {'NB Acc':>8} | {'LR Acc':>8} | {'Winner':>8}")
print("-" * 52)

for ds_name, (data_x, data_y, dtype) in comparisons.items():
    if dtype == "text":
        nb_pipe = Pipeline([("tfidf", TfidfVectorizer(max_features=500)),
                            ("clf", MultinomialNB())])
        lr_pipe = Pipeline([("tfidf", TfidfVectorizer(max_features=500)),
                            ("clf", LogisticRegression(max_iter=300))])
    else:
        nb_pipe = Pipeline([("sc", MinMaxScaler()), ("clf", GaussianNB())])
        lr_pipe = Pipeline([("sc", MinMaxScaler()),
                            ("clf", LogisticRegression(max_iter=300))])
    nb_acc = cross_val_score(nb_pipe, data_x, data_y,
                              cv=cv_comp, scoring="accuracy").mean()
    lr_acc = cross_val_score(lr_pipe, data_x, data_y,
                              cv=cv_comp, scoring="accuracy").mean()
    winner = "NB" if nb_acc >= lr_acc else "LR"
    print(f"{ds_name:>20} | {nb_acc:>8.4f} | {lr_acc:>8.4f} | {winner:>8}")
